# Personalizando o atendimento com Agentes

LLMs são ótimos para responder a perguntas gerais. No entanto, isso sozinho não é suficiente para fornecer valor aos seus clientes.

Para ser capaz de fornecer respostas mais complexas, informações adicionais e específicas para o usuário são necessárias, como seu ID de contrato, o último e-mail que enviaram para o seu suporte ou seu relatório de compras mais recentes.

Agentes são projetados para superar este desafio. Eles são implantações de IA mais avançadas, compostas por múltiplas entidades (ferramentas) especializadas em diferentes ações (recuper informações ou interagir com sistemas externos).

De forma geral, você constrói e apresenta um conjunto de funções personalizadas para a IA. A LLM pode então raciocinar sobre quais informações precisam ser reunidas e quais ferramentas utilizar para responder às intruções recebidas.

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-tools-functions/llm-tools-functions-flow.png?raw=true" width="100%">

# Configuração do ambiente

Vamos começar selecionando o nosso banco de dados

In [0]:
%sql USE vr_demo.playground

# Introdução

## Foundation Models

<img src="https://docs.databricks.com/en/_images/serving-endpoints-list.png" style="float: right; padding-left: 10px; padding-top: 15px" width=600>

Precisamos de um modelo capaz de interpretar o texto das avaliações e extrair as informações desejadas. Para isso, vamos utilizar **[Foundation Models](https://docs.databricks.com/en/machine-learning/foundation-models/index.html#pay-per-token-foundation-model-apis)**, que são grandes modelos de linguagem (LLMs) servidos pela Databricks e que podem ser consultados sob-demanda sem a necessidade de implantação ou gerenciamento desses recursos.

Alguns modelos disponíveis são:

- Anthropic Claude 3.7 Sonnet
- Llama 3.3 70B Instruct
- Llama 3.1 405B Instruct
- DBRX Instruct
- Mixtral-8x7B Instruct
- GTE Large
- BGE Large

Agora, vamos vê-los em funcionamento!

1. No **menu principal** à esquerda, clique em **`Serving`**
2. No card do modelo **Meta Llama 3.3 70B Instruct**, clique em **`Use`**
3. Adicione a instrução abaixo:
    ```
    Classifique o sentimento da seguinte avaliação:
    Comprei um tablet e estou muito insatisfeito com a qualidade da bateria. Ela dura muito pouco tempo e demora muito para carregar.
    ```
    <br>
4. Clique no ícone **enviar**

Com isso, já conseguimos de forma rápida começar a prototipar nossos novos produtos de dados!

## AI Playground

<img src="https://docs.databricks.com/en/_images/ai-playground.gif" style="float: right; padding-left: 10px" width=600>

Para decidir qual o melhor modelo e instrução para o nosso caso de uso, podemos utilizar o **[AI Playground](https://docs.databricks.com/en/large-language-models/ai-playground.html)**.

Assim, podemos testar rapidamente diversas combinações de modelos e instruções através de uma interface intuitiva e escolher a melhor opção par utilizarmos no nosso projeto.

Vamos fazer o seguinte teste:

1. No **menu principal** à esquerda, clique em **`Playgroud`**
2. Clique no **seletor de modelos** e selecione o modelo **`Meta Llama 3.1 70B Instruct`** (caso já não esteja selecionado)
3. Clique no ícone **`Add endpoint`**
4. Clique no **seletor de modelos** e selecione o modelo **`DBRX Instruct`**
5. Clique no ícone **`Add endpoint`**
6. Clique no **seletor de modelos** e selecione o modelo **`Mixtral-8x7B Instruct`**
7. Adicione a instrução abaixo:
    ```
    Classifique o sentimento da seguinte avaliação:
    Comprei um tablet e estou muito insatisfeito com a qualidade da bateria. Ela dura muito pouco tempo e demora muito para carregar.
    ```
    <br>
8. Clique no ícone **enviar**

Agora, podemos comparar as respostas, o tempo e o custo de cada um dos modelos para escolher aquele que melhor atende às necessidades do nosso projeto!

# Definindo as ferramentas

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-tools-functions/llm-tools-functions-playground.gif?raw=true" style="float: right; padding-left: 10px" width=600>

As ferramentas de agentes de IA permitem que os agentes realizem tarefas além da geração de linguagem, como recuperar dados estruturados ou não estruturados e executar código personalizado.

Para criar uma ferramenta com o Mosaic AI Agent Framework, você pode usar qualquer combinação dos seguintes métodos:

|Método| Descrição|
|---|---|
|**Funções do Unity Catalog**| - Definidas e gerenciadas no Unity Catalog com recursos de segurança e conformidade integrados <br> - Cria um registro central para ferramentas que podem ser governadas como outros objetos do Unity Catalog <br> - Concede maior facilidade de descoberta e reutilização <br> - Ideal para aplicar transformações e agregações em grandes conjuntos de dados|
|**Ferramentas de código de agente**| - Definidas no código do agente de IA <br> - Úteis para chamar APIs REST, usar código arbitrário ou executar ferramentas de baixa latência <br> - Não possuem governança integrada e facilidade de descoberta de funções|

## Executando processamentos arbitrários

Ferramentas podem ser muito úteis para definir as tarefas que um agente pode executar. Muitas vezes, essas tarefas são específicas do nosso negócio e precisamos definir como o agente irá executá-las. Nesse casos, podemos usar uma lógica arbitrária para desenvolver essas rotinas de forma mais flexível.

Alguns casos de uso são:
* Cálculos matemáticos
* Tratamentos de texto
* Aplicação de regras de negócio
* Validações

In [0]:
%sql CREATE OR REPLACE FUNCTION valida_cpf(
  cpf STRING COMMENT 'Número do CPF'
)
RETURNS BIGINT
LANGUAGE PYTHON
COMMENT 'Use esta função para validar um CPF e convertê-lo para número. Retorna -1 se o CPF for inválido.'
AS
$$
  cpf = cpf.replace(".", "").replace("-", "")
  if len(cpf) != 11:
    return False
  elif not cpf.isdigit():
    return False
  else:
    d1 = (int(cpf[0])*1 + int(cpf[1])*2 + int(cpf[2])*3 + int(cpf[3])*4 + int(cpf[4])*5 + int(cpf[5])*6 + int(cpf[6])*7 + int(cpf[7])*8 + int(cpf[8])*9) % 11 % 10
    d2 = (int(cpf[0])*0 + int(cpf[1])*1 + int(cpf[2])*2 + int(cpf[3])*3 + int(cpf[4])*4 + int(cpf[5])*5 + int(cpf[6])*6 + int(cpf[7])*7 + int(cpf[8])*8 + d1*9) % 11 % 10
    if d1 == int(cpf[9]) and d2 == int(cpf[10]):
      return int(cpf)
    else:
      return -1
$$

In [0]:
%sql SELECT valida_cpf("111.111.111-11")

## Consultando dados estruturados

Para conseguirmos extrair o máximo de valor dos nossos agentes, precisamos que eles consigam acessar nossos dados corporativos. Somente com essa combinação conseguiremos de fato criar aplicações capazes de impactar o nosso negócio.

### Acessando dados do Lakehouse


Vamos ver como acessar os dados das nossas tabelas Delta.

In [0]:
%sql CREATE OR REPLACE FUNCTION consultar_cliente(qid BIGINT)
RETURNS TABLE (id_cliente BIGINT, nome STRING, sobrenome STRING, num_pedidos INT)
COMMENT 'Use esta função para consultar os dados de um cliente'
RETURN SELECT id_cliente, nome, sobrenome, num_pedidos FROM vr_demo.playground.clientes c WHERE c.id_cliente = qid

In [0]:
%sql SELECT * FROM consultar_cliente(11111111111)

### Online Tables

<img src="https://docs.databricks.com/aws/en/assets/images/create-online-table-473e834357fdf2f576706b4a9100850f.png" style="float: right; width: 800px; margin-left: 10px">

Para cenários onde necessitamos de menores latências, também podemos utilizar **Databricks Online Tables**, que são uma cópia somente de leitura de uma tabela Delta que é armazenada em formato orientado a linhas, otimizado para acesso **online**.

Online Tables são totalmente **serverless** e escalam automaticamente a capacidade de throughput conforme a carga de solicitações, proporcionando baixa latência e alto throughput no acesso a dados de qualquer escala.

Online Tables fornecem **integração** com o Mosaic AI Model Serving, Feature Serving e aplicações de geração aumentada por recuperação (RAG), onde são usadas para consultas rápidas de dados.

## Consultando dados não-estruturados

<img src="https://www.databricks.com/sites/default/files/2024-01/db-vector-search-image-01_0.png?v=1705100714" style="float: right; width: 800px; margin-left: 10px">

No entanto, muitas vezes os dados que precisamos acessar não são necessariamente estruturados ou não estamos querendo fazer uma busca exata.

O **Databricks Vector Search** é um banco de dados vetorial serverless, **integrado** de forma transparente na Data Intelligence Platform.

Ao contrário de outros bancos de dados, o Databricks Vector Search suporta a **sincronização automática** de dados da fonte para o índice, eliminando a manutenção complexa e cara de pipelines.

Ele aproveita as mesmas ferramentas de **segurança e governança** de dados que as organizações já construíram para maior tranquilidade.

Com seu design serverless, o Databricks Vector Search **escala** facilmente para suportar bilhões de embeddings e milhares de consultas em tempo real por segundo.

### Consultando um FAQ

Bases de conhecimento, como FAQs, scripts de atendimento e regras de compliance, podem ser facilmente indexadas com o Databricks Vector Search. Com isso, podemos:

* Identificar documentos relevantes
* Aumentar a acuracidade das respostas dos modelos de IA Generativa 
* Sem necessidade de pré-treinar ou refinar estes modelos

In [0]:
%sql CREATE OR REPLACE FUNCTION consultar_faq(pergunta STRING)
RETURNS TABLE(id LONG, pergunta STRING, resposta STRING, search_score DOUBLE)
COMMENT 'Use esta função para consultar a base de conhecimento sobre prazos de entrega, pedidos de troca ou devolução, entre outras perguntas frequentes sobre o nosso marketplace'
RETURN select * from vector_search(
  index => 'vr_demo.playground.faq_index', 
  query => consultar_faq.pergunta,
  num_results => 1
)

In [0]:
%sql SELECT * FROM consultar_faq("Como fazer uma devolução?")

### Consultando produtos similares

Outro cenário interessante também é o de pesquisas por similaridade. 

Por exemplo, um usuário pode estar buscando alguma característica específica do produto que não esteja categorizada nos filtros existentes do nosso website.

Dessa forma, podemos pesquisar dentro das descrições dos produtos e encontrar aqueles com maior relevância para o cliente, facilitando a descoberta destes produtos e aumentando as chances de conversão.

In [0]:
%sql CREATE OR REPLACE FUNCTION buscar_produtos_semelhantes(descricao STRING)
RETURNS TABLE(id LONG, produto STRING, descricao STRING, search_score DOUBLE)
COMMENT 'Esta função recebe a descrição de um produto, que é utilizada para buscar produtos semelhantes'
RETURN SELECT * FROM vector_search(
  index => 'vr_demo.playground.produtos_index',
  query => buscar_produtos_semelhantes.descricao,
  num_results => 10)
WHERE search_score BETWEEN 0.003 AND 0.99
LIMIT 3

In [0]:
%sql SELECT * FROM buscar_produtos_semelhantes("O fone de ouvido DEF é um dispositivo de áudio projetado para fornecer uma experiência de som imersiva e de alta qualidade. Com drivers de alta fidelidade e tecnologia de cancelamento de ruído, ele permite que você se perca na música ou nos detalhes de um podcast sem distrações. Além disso, seu design ergonômico garante confort durante o uso prolongado.")

## Consultando Foundation Models com prompt engineering

Para customizar o comportamento dos nossos modelos de IA Generativa, podemos utilizar prompts curados por especialistas. Dessa forma, conseguimos:

* Aproveitar melhor o conhecimento de especialistas em IA ou determinado domínio de negócio para aumentar a **eficiência** dos agentes
* Promover a **reutilização** desses ativos entre projetos 
* **Democratizar** o acesso a IA para usuários menos avançados

### Revisão de avaliações

Nosso objetivo é permitir a análise rápida de grandes volumes de avaliações de forma rápida e eficiente. Para isso, precisamos extrair as seguintes informações:

- Produtos mencionados
- Sentimento do cliente
- Caso seja negativo, qual o motivo da insatisfação

Vamos ver como podemos aplicar IA Generativa para acelerar nosso trabalho.

In [0]:
%sql CREATE OR REPLACE FUNCTION REVISAR_AVALIACAO(avaliacao STRING)
RETURNS STRUCT<produto_nome: STRING, produto_categoria: STRING, sentimento: STRING, resposta: STRING, resposta_motivo: STRING>
RETURN FROM_JSON(
  AI_QUERY(
    'databricks-meta-llama-3-1-70b-instruct',
    CONCAT(
      'Um cliente fez uma avaliação. Nós respondemos todos que aparentem descontentes.
      Extraia as seguintes informações:
      - extraia o nome do produto
      - extraia a categoria de produto, por exemplo: tablet, notebook, smartphone
      - classifique o sentimento como ["POSITIVO","NEGATIVO","NEUTRO"]
      - retorne se o sentimento é NEGATIVO e precisa de responsta: S ou N
      - se o sentimento é NEGATIVO, explique quais os principais motivos
      Retorne somente um JSON. Nenhum outro texto fora o JSON. Formato do JSON:
      {
        "produto_nome": <entidade nome>,
        "produto_categoria": <entidade categoria>,
        "sentimento": <entidade sentimento>,
        "resposta": <S ou N para resposta>,
        "motivo": <motivos de insatisfação>
      }
      Avaliação: ', avaliacao
    )
  ),
  "STRUCT<produto_nome: STRING, produto_categoria: STRING, sentimento: STRING, resposta: STRING, motivo: STRING>"
)

In [0]:
%sql SELECT revisar_avaliacao("Comprei um tablet e estou muito insatisfeito com a qualidade da bateria. Ela dura muito pouco tempo e demora muito para carregar.")

### Personalização de respostas

Com todas as informações extraídas, podemos aproveitá-las para gerar sugestões de respostas personalizadas para acelerar o trabalho dos nossos times de atendimento.

Outro ponto interessante é que, nesse processo, podemos aproveitar outras **informações estruturadas** que já tenhamos no nosso ambiente, como dados demográficos, psicográficos e o histórico de compras, para customizar ainda mais nossas respostas!

Vamos ver como fazer isso!

In [0]:
%sql CREATE OR REPLACE FUNCTION GERAR_RESPOSTA(nome STRING, sobrenome STRING, num_pedidos INT, produto STRING, motivo STRING)
RETURNS TABLE(resposta STRING)
COMMENT 'Caso o cliente demonstre insatisfação com algum produto, use esta função para gerar uma resposta personalizada'
RETURN SELECT AI_QUERY(
    'databricks-meta-llama-3-1-70b-instruct',
    CONCAT(
        "Você é um assistente virtual de um e-commerce. Nosso cliente, ", gerar_resposta.nome, " ", gerar_resposta.sobrenome, " que comprou ", gerar_resposta.num_pedidos, " produtos este ano estava insatisfeito com o produto ", gerar_resposta.produto, 
        ", pois ", gerar_resposta.motivo, ". Forneça uma breve mensagem empática para o cliente incluindo a oferta de troca do produto, caso  esteja em conformidade com a nossa política de trocas. A troca pode ser feita diretamente por esse assistente. ",
        "Eu quero recuperar sua confiança e evitar que ele deixe de ser nosso cliente. ",
        "Escreva uma mensagem com poucas sentenças. ",
        "Não adicione nenhum texto além da mensagem. ",
        "Não adicione nenhuma assinatura."
    )
)

In [0]:
%sql SELECT * FROM gerar_resposta("João", "Silva", 23, "tablet DEF", "duração da bateria")

## Consultando uma Genie

A **Databricks AI/BI Genie** permite criar uma interface conversacional para que qualquer tipo de usuário consiga interagir com dados corporativos através de linguagem natural.

Para os nossos agentes, pode ser muito interessante aproveitar esse recurso para integrar esse tipo de consulta em seus workflows.

Isso irá permitir respostas arbitrárias dos usuários sem a necessidade de pré-definir diversas consultas para cada cenário possível, bem como evitar a necessidade de aplicar técnicas complexas para atingir esses resultados.

### Função auxiliar

Normalmente, Genie Spaces são organizados de acordo domínios ou assuntos de negócio, de tal forma que podemos querer acessar mais de uma Genie a partir dos nossos agentes.

Para facilitar a criação das nossas ferramentas para cada Genie, vamos criar uma função auxiliar com a lógica dessa integração.

In [0]:
def aux_genie(databricks_host: str, databricks_token: str, space_id: str, question: str, contextual_history: str) -> str:
  """
  Função auxiliar para integração com a Genie.
  """

  import json
  import os
  import time
  from dataclasses import dataclass
  from datetime import datetime
  from typing import Optional
  import pandas as pd
  import requests

  @dataclass
  class GenieResult:
      space_id: str
      conversation_id: str
      question: str
      content: Optional[str]
      sql_query: Optional[str] = None
      sql_query_description: Optional[str] = None
      sql_query_result: Optional[pd.DataFrame] = None
      error: Optional[str] = None

      def to_json_results(self):
          result = {
              "space_id": self.space_id,
              "conversation_id": self.conversation_id,
              "question": self.question,
              "content": self.content,
              "sql_query": self.sql_query,
              "sql_query_description": self.sql_query_description,
              "sql_query_result": self.sql_query_result.to_dict(
                  orient="records") if self.sql_query_result is not None else None,
              "error": self.error,
          }
          jsonified_results = json.dumps(result)
          return f"Genie Results are: {jsonified_results}"

      def to_string_results(self):
          results_string = self.sql_query_result.to_dict(orient="records") if self.sql_query_result is not None else None
          return ("Genie Results are: \n"
                  f"Space ID: {self.space_id}\n"
                  f"Conversation ID: {self.conversation_id}\n"
                  f"Question That Was Asked: {self.question}\n"
                  f"Content: {self.content}\n"
                  f"SQL Query: {self.sql_query}\n"
                  f"SQL Query Description: {self.sql_query_description}\n"
                  f"SQL Query Result: {results_string}\n"
                  f"Error: {self.error}")

  class GenieClient:

      def __init__(self, *,
                    host: Optional[str] = None,
                    token: Optional[str] = None,
                    api_prefix: str = "/api/2.0/genie/spaces"):
          self.host = host or os.environ.get("DATABRICKS_HOST")
          self.token = token or os.environ.get("DATABRICKS_TOKEN")
          assert self.host is not None, "DATABRICKS_HOST is not set"
          assert self.token is not None, "DATABRICKS_TOKEN is not set"
          self._workspace_client = requests.Session()
          self._workspace_client.headers.update({"Authorization": f"Bearer {self.token}"})
          self._workspace_client.headers.update({"Content-Type": "application/json"})
          self.api_prefix = api_prefix
          self.max_retries = 300
          self.retry_delay = 1
          self.new_line = "\r\n"

      def _make_url(self, path):
          return f"{self.host.rstrip('/')}/{path.lstrip('/')}"

      def start(self, space_id: str, start_suffix: str = "") -> str:
          path = self._make_url(f"{self.api_prefix}/{space_id}/start-conversation")
          resp = self._workspace_client.post(
              url=path,
              headers={"Content-Type": "application/json"},
              json={"content": "starting conversation" if not start_suffix else f"starting conversation {start_suffix}"},
          )
          resp = resp.json()
          print(resp)
          try:
            return resp["conversation_id"]
          except Exception:
            return resp

      def ask(self, space_id: str, conversation_id: str, message: str) -> GenieResult:
          path = self._make_url(f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages")
          # TODO: cleanup into a separate state machine
          resp_raw = self._workspace_client.post(
              url=path,
              headers={"Content-Type": "application/json"},
              json={"content": message},
          )
          resp = resp_raw.json()
          message_id = resp.get("message_id", resp.get("id"))
          if message_id is None:
              print(resp, resp_raw.url, resp_raw.status_code, resp_raw.headers)
              return GenieResult(content=None, error="Failed to get message_id")

          attempt = 0
          query = None
          query_description = None
          content = None

          while attempt < self.max_retries:
              resp_raw = self._workspace_client.get(
                  self._make_url(f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}"),
                  headers={"Content-Type": "application/json"},
              )
              resp = resp_raw.json()
              status = resp["status"]
              if status == "COMPLETED":
                  try:

                      query = resp["attachments"][0]["query"]["query"]
                      query_description = resp["attachments"][0]["query"].get("description", None)
                      content = resp["attachments"][0].get("text", {}).get("content", None)
                  except Exception as e:
                      return GenieResult(
                          space_id=space_id,
                          conversation_id=conversation_id,
                          question=message,
                          content=resp["attachments"][0].get("text", {}).get("content", None)
                      )
                  break

              elif status == "EXECUTING_QUERY":
                  self._workspace_client.get(
                      self._make_url(
                          f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}/query-result"),
                      headers={"Content-Type": "application/json"},
                  )
              elif status in ["FAILED", "CANCELED"]:
                  return GenieResult(
                      space_id=space_id,
                      conversation_id=conversation_id,
                      question=message,
                      content=None,
                      error=f"Query failed with status {status}"
                  )
              elif status != "COMPLETED" and attempt < self.max_retries - 1:
                  time.sleep(self.retry_delay)
              else:
                  return GenieResult(
                      space_id=space_id,
                      conversation_id=conversation_id,
                      question=message,
                      content=None,
                      error=f"Query failed or still running after {self.max_retries * self.retry_delay} seconds"
                  )
              attempt += 1
          resp = self._workspace_client.get(
              self._make_url(
                  f"{self.api_prefix}/{space_id}/conversations/{conversation_id}/messages/{message_id}/query-result"),
              headers={"Content-Type": "application/json"},
          )
          resp = resp.json()
          columns = resp["statement_response"]["manifest"]["schema"]["columns"]
          header = [str(col["name"]) for col in columns]
          rows = []
          output = resp["statement_response"]["result"]
          if not output:
              return GenieResult(
                  space_id=space_id,
                  conversation_id=conversation_id,
                  question=message,
                  content=content,
                  sql_query=query,
                  sql_query_description=query_description,
                  sql_query_result=pd.DataFrame([], columns=header),
              )
          for item in resp["statement_response"]["result"]["data_typed_array"]:
              row = []
              for column, value in zip(columns, item["values"]):
                  type_name = column["type_name"]
                  str_value = value.get("str", None)
                  if str_value is None:
                      row.append(None)
                      continue
                  match type_name:
                      case "INT" | "LONG" | "SHORT" | "BYTE":
                          row.append(int(str_value))
                      case "FLOAT" | "DOUBLE" | "DECIMAL":
                          row.append(float(str_value))
                      case "BOOLEAN":
                          row.append(str_value.lower() == "true")
                      case "DATE":
                          row.append(datetime.strptime(str_value, "%Y-%m-%d").date())
                      case "TIMESTAMP":
                          row.append(datetime.strptime(str_value, "%Y-%m-%d %H:%M:%S"))
                      case "BINARY":
                          row.append(bytes(str_value, "utf-8"))
                      case _:
                          row.append(str_value)
              rows.append(row)

          query_result = pd.DataFrame(rows, columns=header)
          return GenieResult(
              space_id=space_id,
              conversation_id=conversation_id,
              question=message,
              content=content,
              sql_query=query,
              sql_query_description=query_description,
              sql_query_result=query_result,
          )


  assert databricks_host is not None, "host is not set"
  assert databricks_token is not None, "token is not set"
  assert space_id is not None, "space_id is not set"
  assert question is not None, "question is not set"
  assert contextual_history is not None, "contextual_history is not set"
  client = GenieClient(host=databricks_host, token=databricks_token)
  conversation_id = client.start(space_id)
  if isinstance(conversation_id, str) is False:
      return conversation_id
  formatted_message = f"""Use the contextual history to answer the question. The history may or may not help you. Use it if you find it relevant.

  Contextual History: {contextual_history}

  Question to answer: {question}
  """

  try:
      result = client.ask(space_id, conversation_id, formatted_message)
  except Exception as e:
      return f"Error: {str(e)}"
  return result.to_string_results()

In [0]:
aux_genie(
  dbutils.secrets.get('my_secret_scope', 'vr-host'),
  dbutils.secrets.get('my_secret_scope', 'vr-genie-secret'),
  '01f058479ad31fd0b21551c3ce350db9',
  'Quantos produtos foram vendidos em out/22?', 
  'Sem histórico'
)

Para quem prefere trabalhar exclusimante com Python, podemos utilizar o **DatabricksFunctionClient** para registrar nossas funções.

In [0]:
%pip install unitycatalog-ai[databricks] unitycatalog-langchain[databricks] databricks-langchain
dbutils.library.restartPython()

In [0]:
%sql USE vr_demo.playground

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
client = DatabricksFunctionClient()
info = client.create_python_function(func=aux_genie, catalog='vr_demo', schema='playground', replace=True)

### Genie de vendas

Agora, vamos criar uma ferramenta específica para interagir com os dados de vendas.

In [0]:
%sql CREATE OR REPLACE FUNCTION conversar_com_vendas(
  question STRING COMMENT 'Pergunta a ser respondida pela Genie.',
  contextual_history STRING COMMENT 'Forneça o histórico relevante para responder a pergunta. Assuma que a Genie não armazena o histórico. Use "nenhum histórico relevante" se não houver nada relevante para responder a questão.'
)
RETURNS STRING
COMMENT 'Use esta função para fazer perguntas relacionadas a venda de produtos.'
RETURN

SELECT aux_genie(
  secret('my_secret_scope', 'vr-host'),
  secret('my_secret_scope', 'vr-genie-secret'),
  '01f058479ad31fd0b21551c3ce350db9',
  question,
  contextual_history
)

In [0]:
%sql SELECT conversar_com_vendas('Quantos produtos foram vendidos em out/22?', '')

## Consultando uma API

Muitas vezes, além de acessar nossos dados internos, também precisamos consultar APIs externas para executar determinado processo.

Isso nos permite **integrar** com sistemas de análise de crédito, geolocalização e etc em tempo real.

Além disso, também podemos usar este recurso para **disparar ações** em nossos sistemas automaticamente, como o envio de alertas, abertura de ordens de compras / serviço ou criação de tickets para avaliação por um humano.

Aqui, vamos utilizar recursos básicos do Python para consultar uma API aberta.

In [0]:
%sql CREATE OR REPLACE FUNCTION consultar_clima(latitude DOUBLE, longitude DOUBLE)
RETURNS STRUCT<temperature_in_celsius DOUBLE, rain_in_mm DOUBLE>
LANGUAGE PYTHON
COMMENT 'Esta função consulta a temperatura atual e informações sobre chuva para uma determinada latitude e longitude usando a API Open-Meteo.'
AS
$$
  import requests as r
  weather = r.get(f'https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,rain&forecast_days=1').json()
  return {
    "temperature_in_celsius": weather["current"]["temperature_2m"],
    "rain_in_mm": weather["current"]["rain"]
  }
$$

In [0]:
%sql SELECT consultar_clima(-23.6811632,-47.2191291)

## Enviando notificações

Vamos ver como utilizar o Telegram para enviar mensagens automáticas para os nossos clientes.

### Criando uma Unity Catalog Connection

Dessa vez, utilizaremos uma Unity Catalog Connection para gerenciar as informações de acesso, credenciais e permissões de utilização na plataforma.

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS vr_telegram TYPE HTTP
OPTIONS (
  host 'https://api.telegram.org',
  base_path '/',
  bearer_token 'token'
)

### Enviar Telegram

Vamos utilizar essa conexão para enviar uma mensagem para o cliente através do Telegram.

In [0]:
%sql CREATE OR REPLACE FUNCTION enviar_telegram(
  text STRING COMMENT 'Texto da mensagem a ser enviada para o Telegram.'
)
RETURNS STRING
COMMENT 'Use esta função para enviar mensagens ao Telegram.'
RETURN

SELECT CASE 
  WHEN (
    http_request(
      conn => 'vr_telegram',
      method => 'POST',
      path => CONCAT('bot', secret('my_secret_scope', 'vr-telegram-secret'), '/sendMessage'),
      json => to_json(named_struct(
        'chat_id', '7337562046',
        'text', text
      ))
    )).status_code = 200
  THEN 'Mensagem enviada com sucesso!'
  ELSE 'Falha no envio da mensagem!'
  END

In [0]:
%sql SELECT enviar_telegram('Olá! Como posso ajudá-lo?')

# Avaliando o agente no AI Playground

<img src="https://docs.databricks.com/en/_images/ai-playground.gif" style="float: right; padding-left: 10px" width=600>

Para decidir qual o melhor modelo e instrução para o nosso caso de uso, podemos utilizar o **[AI Playground](https://docs.databricks.com/en/large-language-models/ai-playground.html)**.

Assim, podemos testar rapidamente diversas combinações de modelos e instruções através de uma interface intuitiva e escolher a melhor opção par utilizarmos no nosso projeto.

Vamos fazer o seguinte teste:

1. No **menu principal** à esquerda, clique em **`Playgroud`**
1. Clique no **seletor de modelos** e selecione o modelo **`Meta Llama 3.3 70B Instruct`** (caso já não esteja selecionado)
1. Clique no ícone **`Add endpoint`**
1. Adicione a instrução abaixo:<br>
    `Você é um assistente virtual de um e-commerce. Para responder à perguntas, é necessário que o cliente forneça um CPF válido. Caso ainda não tenha essa informação, solicite o CPF educadamente. Após validar o CPF, lembre-se de consultar os dados do cliente para personalizar suas respostas. Caso o CPF do cliente não exista na nossa base, peça educadamente um novo CPF. Você pode responder perguntas sobre entrega, devolução de produtos, status de pedidos, entre outros. Se você não souber como responder a pergunta, diga que você não sabe. Não invente ou especule sobre nada. Sempre que perguntado sobre procedimentos, consulte nossa base de conhecimento.`
    <br>
1. Clique em **`Tools`** > **`Add tool`**
1. Digite o caminho do seu **database** como segue: `{catalogo}.{database}.*`
1. Clique no ícone **enviar**

Agora, podemos avaliar as respostas, o tempo e o custo do nosso agente para entender se ele atende às necessidades do nosso projeto!